In [1]:
import pandas as pd
from datetime import datetime, timezone, timedelta
import csv
import os
import requests
import json
import time

- 함수 모은거

In [2]:
API_KEY=""
HEADERS = {"X-Riot-Token": API_KEY}

REGION_MAPPING = {
    'KR': 'asia', 'JP1': 'asia',
    'BR1': 'americas', 'NA1': 'americas', 'LA1': 'americas', 'LA2': 'americas',
    'EUN1': 'europe', 'EUW1': 'europe', 'TR1': 'europe', 'RU': 'europe', 'ME1': 'europe',
    'OC1': 'sea', 'SG2': 'sea', 'TW2': 'sea', 'VN2': 'sea'
}

root_path = r''
data_path = os.path.join(root_path, '0_data')
OUTPUT_FILE = os.path.join(data_path, '16_matchids', 'collected_match_ids_fienn.csv')
PROGRESS_FILE = os.path.join(data_path, '16_matchids', 'processed_puuids_fienn.txt')
SAVE_INTERVAL = 100 

def get_matches_for_one_user(puuid, platform, total_count=1000):
    routing = REGION_MAPPING.get(platform.upper())
    url = f"https://{routing}.api.riotgames.com/lol/match/v5/matches/by-puuid/{puuid}/ids"
    
    user_match_ids = []
    # 수집하고 싶은 큐 ID 리스트
    target_queues = [420, 440] 

    for q_id in target_queues:
        start_index = 0
        print(f"   - Queue {q_id} 수집 시작...", end=" ")
        
        while start_index < total_count:
            params = {
                "queue": q_id,
                "start": start_index,
                "count": 100
            }
            try:
                response = requests.get(url, headers=HEADERS, params=params)
                
                if response.status_code == 200:
                    fetched_ids = response.json()
                    if not fetched_ids: 
                        break
                    
                    user_match_ids.extend(fetched_ids)
                    
                    if len(fetched_ids) < 100: 
                        break
                    
                    start_index += 100
                    time.sleep(0.05) # 개인 키 속도 제한 준수
                    
                elif response.status_code == 429:
                    retry_after = int(response.headers.get("Retry-After", 10))
                    print(f"\n 429 Limit. {retry_after}초 대기 후 Queue {q_id} (Index {start_index}) 재시도...")
                    time.sleep(retry_after + 1)
                    continue # 인덱스 증가 없이 다시 시도
                    
                else:
                    print(f"\n Error {response.status_code} at Queue {q_id}")
                    break
                    
            except Exception as e:
                print(f"\n 연결 오류: {e}. 5초 후 재시도...")
                time.sleep(5)
                continue
        
        print(f"완료.")
            
    # 솔로랭크와 자유랭크 중복 매치가 있을 수 있으므로 set으로 변환했다가 리스트로 반환
    return list(set(user_match_ids))

def save_to_csv(match_id_set, file_name):
    """
    수집된 매치 ID 세트를 CSV에 저장 (중복 제거 포함)
    """
    new_df = pd.DataFrame(list(match_id_set), columns=['match_id'])
    
    # 파일이 없으면 새로 만들고(header 포함), 있으면 이어 쓰기(header 제외)
    if not os.path.exists(file_name):
        new_df.to_csv(file_name, index=False, mode='w', encoding='utf-8')
    else:
        new_df.to_csv(file_name, index=False, mode='a', header=False, encoding='utf-8')
    
    print(f"--- [중간 저장 완료] {len(new_df)}개의 새로운 매치 ID 저장됨 ---")

def collect_with_checkpoints(df_dict):
    current_session_matches = set() # 현재 세션에서의 중복 방지용
    
    for platform, df in df_dict.items():
        print(f"\n🚀 {platform} 지역 수집 시작...")
        
        for idx, puuid in enumerate(df['puuid']):
            m_ids = get_matches_for_one_user(puuid, platform)
            current_session_matches.update(m_ids)
            
            # 중간 저장 로직: SAVE_INTERVAL(10명) 마다 실행
            if (idx + 1) % SAVE_INTERVAL == 0:
                print(f"[{idx+1}/{len(df)}] 진행 중...", end=' ')
                save_to_csv(current_session_matches, OUTPUT_FILE)
                # 저장 후 세션을 비워주면 메모리 부담이 줄어듭니다.
                current_session_matches.clear() 

        # 지역 하나가 끝나면 남은 데이터 저장
        if current_session_matches:
            save_to_csv(current_session_matches, OUTPUT_FILE)
            current_session_matches.clear()

    # 마지막으로 전체 파일에서 혹시 모를 중복 제거
    final_cleanup(OUTPUT_FILE)

def final_cleanup(file_name):
    print("\n🧹 전체 데이터 중복 제거 작업 시작...")
    df = pd.read_csv(file_name)
    before_count = len(df)
    df.drop_duplicates(subset=['match_id'], inplace=True)
    df.to_csv(file_name, index=False)
    print(f"✅ 최종 완료! ({before_count} -> {len(df)}개)")

# # 실행
# # collect_with_checkpoints(df_dict)

def load_processed_puuids():
    """이미 수집 완료된 유저 목록을 불러옵니다."""
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE, 'r') as f:
            # 한 줄씩 읽어서 세트에 저장 (중복 제거 및 빠른 검색용)
            return set(line.strip() for line in f)
    return set()

def save_progress(puuid_list):
    """수집 완료된 유저 목록을 파일에 추가합니다."""
    # 경로가 없으면 생성
    os.makedirs(os.path.dirname(PROGRESS_FILE), exist_ok=True)
    with open(PROGRESS_FILE, 'a') as f:
        for puuid in puuid_list:
            f.write(f"{puuid}\n")

def collect_with_resume(df_dict):
    # 1. 작업 완료 명단 로드
    processed_puuids = load_processed_puuids()
    print(f"📊 기존 작업 기록 확인: {len(processed_puuids)}명의 유저는 이미 수집되었습니다.")

    current_session_matches = set()
    newly_processed_this_session = [] # 이번 세션에서 새로 완료한 유저들

    for platform, df in df_dict.items():
        print(f"\n🚀 {platform} 지역 수집 시작...")
        
        for idx, puuid in enumerate(df['puuid']):
            # 2. 건너뛰기 로직
            if puuid in processed_puuids:
                continue 

            # 3. 데이터 수집
            m_ids = get_matches_for_one_user(puuid, platform, total_count=1000)
            current_session_matches.update(m_ids)
            newly_processed_this_session.append(puuid)
            
            # 4. 중간 저장 (SAVE_INTERVAL 마다)
            if len(newly_processed_this_session) >= SAVE_INTERVAL:
                print(f"[{idx+1}/{len(df)}] 저장 중...", end=' ')
                save_to_csv(current_session_matches, OUTPUT_FILE)
                save_progress(newly_processed_this_session) # 유저 명단도 저장
                
                # 메모리 비우기
                current_session_matches.clear()
                processed_puuids.update(newly_processed_this_session)
                newly_processed_this_session = []

        # 지역 종료 후 남은 데이터 처리
        if newly_processed_this_session:
            save_to_csv(current_session_matches, OUTPUT_FILE)
            save_progress(newly_processed_this_session)
            current_session_matches.clear()
            newly_processed_this_session = []

    print("\n✨ 모든 지역 수집 프로세스가 종료되었습니다.")

## 샘플 유저 데이터 서버마다 합치기

In [3]:
import warnings
warnings.filterwarnings('ignore')

def perform_stratified_sampling(sample_fraction, input_file, output_file, min_per_group=1, stratify_col='tier', random_state=42):
    """
    계층적 샘플링 수행 (기본: 티어별 샘플링).
    - sample_fraction: 각 그룹에서 비율로 뽑을 비율
    - min_per_group: 그룹이 작을 때 최소 보장 샘플 수
    - stratify_col: 계층화에 사용할 컬럼 이름
    """
    print('📂 데이터 로딩 중...')
    df = pd.read_csv(input_file)

    if stratify_col not in df.columns:
        raise ValueError(f"'{stratify_col}' 컬럼이 입력 파일에 없습니다.")

    # 그룹별로 샘플 수 계산
    def _sample_group(g):
        grp_size = len(g)
        n = max(0, int(round(grp_size * sample_fraction)))
        if n < min_per_group and grp_size > 0:
            n = min(min_per_group, grp_size)
        if n <= 0:
            return g.iloc[0:0]
        return g.sample(n=n, random_state=random_state)

    sampled_df = df.groupby(stratify_col, group_keys=False).apply(_sample_group).reset_index(drop=True)

    # 결과 출력 및 저장
    print('\n✅ 샘플링 완료!')
    print(f"전체 데이터 수: {len(df):,} 명")
    print(f"샘플링 데이터 수: {len(sampled_df):,} 명")
    print(f"샘플 내 {stratify_col} 비중:\n{(sampled_df[stratify_col].value_counts(normalize=True) * 100).round(2)}%")

    sampled_df.to_csv(output_file, index=False, encoding='utf-8')
    print(f"💾 샘플 데이터가 {output_file}에 저장되었습니다.")

In [4]:
# 환경에 맞게 파일 경로 입력
file_paths = [
    
]

for name in file_paths:
    '''샘플링 비율은 조정 필요함'''
    sampling = 0.002
    perform_stratified_sampling(
        sample_fraction = sampling,
        input_file=f'{name}.csv', 
        output_file=f'{name}_{sampling*100}%.csv'
    )

📂 데이터 로딩 중...

✅ 샘플링 완료!
전체 데이터 수: 3,684 명
샘플링 데이터 수: 8 명
샘플 내 tier 비중:
tier
MASTER         75.0
CHALLENGER     12.5
GRANDMASTER    12.5
Name: proportion, dtype: float64%
💾 샘플 데이터가 C:\Users\fienn\Documents\GitHub\team_project4\0_data\04_la2_csv\la2_puuids_high_tiers_0.2%.csv에 저장되었습니다.
📂 데이터 로딩 중...

✅ 샘플링 완료!
전체 데이터 수: 380,038 명
샘플링 데이터 수: 759 명
샘플 내 tier 비중:
tier
SILVER      25.69
GOLD        25.16
BRONZE      18.71
PLATINUM    16.86
EMERALD      8.56
DIAMOND      3.29
IRON         1.71
Name: proportion, dtype: float64%
💾 샘플 데이터가 C:\Users\fienn\Documents\GitHub\team_project4\0_data\04_la2_csv\la2_puuids_iron_diamond_0.2%.csv에 저장되었습니다.
📂 데이터 로딩 중...

✅ 샘플링 완료!
전체 데이터 수: 1,387 명
샘플링 데이터 수: 3 명
샘플 내 tier 비중:
tier
CHALLENGER     33.33
GRANDMASTER    33.33
MASTER         33.33
Name: proportion, dtype: float64%
💾 샘플 데이터가 C:\Users\fienn\Documents\GitHub\team_project4\0_data\13_sg2_csv\sg2_puuids_high_tiers_0.2%.csv에 저장되었습니다.
📂 데이터 로딩 중...

✅ 샘플링 완료!
전체 데이터 수: 141,927 명
샘플링 데이터 수: 284 명
샘플 내 

In [3]:
# 샘플링 된 유저데이터 로드
la2_high=pd.read_csv()
sg2_high=pd.read_csv()
jp1_high=pd.read_csv()
tw2_high=pd.read_csv()
ru_high=pd.read_csv()
oc1_high=pd.read_csv()
me1_high=pd.read_csv()

la2_low=pd.read_csv()
sg2_low=pd.read_csv()
jp1_low=pd.read_csv()
tw2_low=pd.read_csv()
ru_low=pd.read_csv()
oc1_low=pd.read_csv()
me1_low=pd.read_csv()

In [4]:
regions = ["LA2", "SG2", "JP1", "TW2", "RU", "OC1", "ME1"]

low_df = [la2_low, sg2_low, jp1_low, tw2_low, ru_low, oc1_low, me1_low]
high_df = [la2_high, sg2_high, jp1_high, tw2_high, ru_high, oc1_high, me1_high]

# 최종적으로 사용할 딕셔너리
df_dict = {}

# zip 활용
for region, low, high in zip(regions, low_df, high_df):
    
    # 컬럼 삭제
    cols_to_drop = ['division', 'leagueId']
    low.drop(columns=cols_to_drop, errors='ignore', inplace=True)
    high.drop(columns=cols_to_drop, errors='ignore', inplace=True)
    
    # Low와 High 합치기
    combined = pd.concat([low, high], ignore_index=True)
    
    # 중복 유저(puuid)가 있을 수 있으니 한 번 더 제거
    combined.drop_duplicates(subset=['puuid'], inplace=True)
    
    # 딕셔너리에 저장 (Key: 서버이름, Value: 통합 데이터프레임)
    df_dict[region] = combined

print(f"✅ {len(df_dict)}개 지역의 데이터 통합 완료!")
print(f"대상 지역: {list(df_dict.keys())}")

✅ 7개 지역의 데이터 통합 완료!
대상 지역: ['LA2', 'SG2', 'JP1', 'TW2', 'RU', 'OC1', 'ME1']


## EXE

In [5]:
collect_with_resume(df_dict)

📊 기존 작업 기록 확인: 0명의 유저는 이미 수집되었습니다.

🚀 LA2 지역 수집 시작...
   - Queue 420 수집 시작... 완료.
   - Queue 440 수집 시작... 완료.
   - Queue 420 수집 시작... 완료.
   - Queue 440 수집 시작... 완료.
   - Queue 420 수집 시작... 완료.
   - Queue 440 수집 시작... 완료.
   - Queue 420 수집 시작... 완료.
   - Queue 440 수집 시작... 완료.
   - Queue 420 수집 시작... 완료.
   - Queue 440 수집 시작... 완료.
   - Queue 420 수집 시작... 완료.
   - Queue 440 수집 시작... 완료.
   - Queue 420 수집 시작... 완료.
   - Queue 440 수집 시작... 완료.
   - Queue 420 수집 시작... 완료.
   - Queue 440 수집 시작... 완료.
   - Queue 420 수집 시작... 완료.
   - Queue 440 수집 시작... 완료.
   - Queue 420 수집 시작... 완료.
   - Queue 440 수집 시작... 완료.
   - Queue 420 수집 시작... 완료.
   - Queue 440 수집 시작... 완료.
   - Queue 420 수집 시작... 완료.
   - Queue 440 수집 시작... 완료.
   - Queue 420 수집 시작... 완료.
   - Queue 440 수집 시작... 완료.
   - Queue 420 수집 시작... 완료.
   - Queue 440 수집 시작... 완료.
   - Queue 420 수집 시작... 완료.
   - Queue 440 수집 시작... 완료.
   - Queue 420 수집 시작... 완료.
   - Queue 440 수집 시작... 완료.
   - Queue 420 수집 시작... 완료.
   - Queue 440 수집 시작..

## 매치데이터

In [6]:
matchids = pd.read_csv(OUTPUT_FILE)

print(matchids.head(), matchids.shape)

         match_id
0  LA2_1525075322
1  LA2_1522942013
2  LA2_1443662524
3  LA2_1497371675
4  LA2_1561894654 (695640, 1)


In [7]:
ch = matchids.copy()
ch['platform']=ch['match_id'].apply(lambda x: x.split('_')[0])
ch.head()

,match_id,platform
0,LA2_1525075322,LA2
1,LA2_1522942013,LA2
2,LA2_1443662524,LA2
3,LA2_1497371675,LA2
4,LA2_1561894654,LA2


In [8]:
ch.groupby('platform').agg('count')

,match_id
platform,
EUN1,368
EUW1,1655
JP1,86585
LA1,492
LA2,301896
ME1,16570
OC1,58563
PH2,12935
RU,58009


In [9]:
ch=ch.loc[ch['platform'].isin(list(df_dict.keys()))]

ch.groupby('platform').agg('count')

,match_id
platform,
JP1,86585
LA2,301896
ME1,16570
OC1,58563
RU,58009
SG2,77564
TW2,77902


In [10]:
len(ch)

677089

In [11]:
ch = ch.sample(frac=0.02, random_state=42)
print(ch.shape)

(13542, 2)


In [12]:
ch=ch.drop(columns=['platform'])

In [13]:
ch.to_csv(os.path.join(data_path, '16_matchids', 'sampled_collected_match_ids_fienn.csv'), index=False)